In [ ]:
!pip install prophet xgboost plotly -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from prophet import Prophet
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import plotly.graph_objects as go
import plotly.io as pio
import warnings
warnings.filterwarnings('ignore')

pio.renderers.default = 'colab'
print("Libraries installed successfully")

In [ ]:
from google.colab import files
print("Upload your CSV file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
print("Data loaded successfully")
print(df.shape)
df.head()

In [ ]:
df['Date'] = pd.to_datetime(df['Order Date'])
df = df.sort_values('Date')

sales_col = 'Sales'

print(df.isnull().sum())

df = df.dropna(subset=[sales_col])

Q1 = df[sales_col].quantile(0.25)
Q3 = df[sales_col].quantile(0.75)
IQR = Q3 - Q1
df = df[(df[sales_col] >= Q1 - 1.5*IQR) & (df[sales_col] <= Q3 + 1.5*IQR)]

print("Cleaned data shape:", df.shape)

In [ ]:
sales_col = 'Sales'

def create_features(df):
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week
    df['Quarter'] = df['Date'].dt.quarter
    df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)

    df['Discount_Pct'] = df['Discount']

    df = pd.get_dummies(df, columns=['Segment', 'City Type', 'Region'], drop_first=True)

    df_daily = df.groupby('Date')[sales_col].sum().reset_index()
    df_daily.columns = ['Date', 'Daily_Sales']

    for lag in [1, 3, 7, 14, 30]:
        df_daily[f'Lag_{lag}'] = df_daily['Daily_Sales'].shift(lag)

    for window in [7, 14, 30]:
        df_daily[f'Rolling_Mean_{window}'] = df_daily['Daily_Sales'].rolling(window).mean()
        df_daily[f'Rolling_Std_{window}'] = df_daily['Daily_Sales'].rolling(window).std()

    df_daily = df_daily.dropna()

    df_daily['Year'] = df_daily['Date'].dt.year
    df_daily['Month'] = df_daily['Date'].dt.month
    df_daily['Day'] = df_daily['Date'].dt.day
    df_daily['DayOfWeek'] = df_daily['Date'].dt.dayofweek
    df_daily['WeekOfYear'] = df_daily['Date'].dt.isocalendar().week
    df_daily['Quarter'] = df_daily['Date'].dt.quarter
    df_daily['IsWeekend'] = (df_daily['DayOfWeek'] >= 5).astype(int)

    return df_daily

df_daily = create_features(df)
print("Features created")
print(df_daily.shape)
df_daily.head()

In [ ]:
sales_col = 'Daily_Sales'

feature_cols = ['Year', 'Month', 'Day', 'DayOfWeek', 'WeekOfYear',
                'Quarter', 'IsWeekend', 'Lag_1', 'Lag_3', 'Lag_7',
                'Lag_14', 'Lag_30', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_30']

X = df_daily[feature_cols]
y = df_daily[sales_col]

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)

xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb.fit(X_train_scaled, y_train)
xgb_pred = xgb.predict(X_test_scaled)

print("Three models trained")

In [ ]:
X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

lstm_model = keras.Sequential([
    layers.LSTM(50, activation='relu', return_sequences=True, input_shape=(1, X_train_scaled.shape[1])),
    layers.Dropout(0.2),
    layers.LSTM(25, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm_model.fit(X_train_lstm, y_train, epochs=15, batch_size=32, verbose=1)
lstm_pred = lstm_model.predict(X_test_lstm).flatten()

print("LSTM trained")

In [ ]:
prophet_train = df_daily[['Date', sales_col]].iloc[:split_idx].rename(columns={'Date': 'ds', sales_col: 'y'})

prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=True)
prophet_model.fit(prophet_train)

prophet_test = df_daily[['Date']].iloc[split_idx:].rename(columns={'Date': 'ds'})
prophet_forecast = prophet_model.predict(prophet_test)
prophet_pred = prophet_forecast['yhat'].values

print("Prophet trained")

In [ ]:
results = {}
models = {
    'Linear Regression': lr_pred,
    'Random Forest': rf_pred,
    'XGBoost': xgb_pred,
    'LSTM': lstm_pred,
    'Prophet': prophet_pred
}

print("Model Performance Comparison")
print("="*65)
print(f"{'Model':<20} | {'MAE':>10} | {'RMSE':>10} | {'R2':>10}")
print("-"*65)

best_r2 = -999
best_model = ""

for name, pred in models.items():
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    if r2 > best_r2:
        best_r2 = r2
        best_model = name

    print(f"{name:<20} | {mae:>10.2f} | {rmse:>10.2f} | {r2:>10.4f}")

print("="*65)
print("Best Model:", best_model)
print("R2 Score:", best_r2)

In [ ]:
last_date = df_daily['Date'].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=180)
future_df = pd.DataFrame({'Date': future_dates})

future_df['Year'] = future_df['Date'].dt.year
future_df['Month'] = future_df['Date'].dt.month
future_df['Day'] = future_df['Date'].dt.day
future_df['DayOfWeek'] = future_df['Date'].dt.dayofweek
future_df['WeekOfYear'] = future_df['Date'].dt.isocalendar().week
future_df['Quarter'] = future_df['Date'].dt.quarter
future_df['IsWeekend'] = (future_df['DayOfWeek'] >= 5).astype(int)

last_known = df_daily[feature_cols].iloc[-1]
for col in ['Lag_1', 'Lag_3', 'Lag_7', 'Lag_14', 'Lag_30',
            'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_30']:
    future_df[col] = last_known[col]

future_scaled = scaler.transform(future_df[feature_cols])
future_predictions = xgb.predict(future_scaled)
future_df['Predicted_Sales'] = future_predictions
future_df['Predicted_Sales'] = future_df['Predicted_Sales'].clip(lower=0)

print("6 months predictions generated")
future_df.head(10)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_daily['Date'],
    y=df_daily['Daily_Sales'],
    mode='lines',
    name='Historical Sales',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=future_df['Date'],
    y=future_df['Predicted_Sales'],
    mode='lines+markers',
    name='Predicted Sales 6 Months',
    line=dict(color='red', dash='dash')
))

fig.update_layout(
    title='6 Month Sales Forecast',
    xaxis_title='Date',
    yaxis_title='Sales',
    hovermode='x unified',
    template='plotly_dark'
)

fig.show()

future_df.to_csv('six_month_forecast.csv', index=False)
print("Forecast saved")

In [ ]:
from google.colab import files
files.download('six_month_forecast.csv')
print("Download complete")